# 05 - Generación de Campaña con IA Local (LM Studio)

En este notebook se realiza la generación masiva de correos electrónicos de fidelización utilizando un modelo de lenguaje de gran escala (LLM) ejecutado en local. 

**Arquitectura:**
- **Modelo:** Llama 3.1 8B (vía LM Studio).
- **Interfaz:** OpenAI-compatible API (localhost:1234).
- **Base de datos:** SQLite (crm_gimnasio.db).

*Nota: Este método sustituye el uso de APIs comerciales para garantizar privacidad, eliminar costes y evitar límites de tokens.*

In [6]:
import sqlite3
import pandas as pd
from openai import OpenAI

# 1. Conexión al servidor local de LM Studio
# Asegúrate de tener el "Local Server" encendido en la app
client = OpenAI(base_url="http://localhost:11434/v1", api_key="lm-studio")

# 2. Selección de un socio aleatorio
conexion = sqlite3.connect('../../datos/crm_gimnasio.db')
socio_aleatorio = pd.read_sql("SELECT * FROM campana_retencion ORDER BY RANDOM() LIMIT 1", conexion).iloc[0]
conexion.close()

id_socio = int(socio_aleatorio['id_socio'])
dias_sin_venir = int(socio_aleatorio['dias_sin_venir'])
zona = int(socio_aleatorio['zona_proximidad'])
visitas = int(socio_aleatorio['total_visitas'])
edad = int(socio_aleatorio['edad'])
cuota = str(socio_aleatorio['tipo_socio'])

# 3. Reglas de negocio y preprocesamiento lógico
# Lógica de distancia
if zona == 0:
    texto_ubicacion = "Vive muy CERCA del gimnasio."
    regla_distancia = "Destaca la comodidad de tener el centro a un paso de casa."
else:
    texto_ubicacion = "Vive LEJOS del gimnasio."
    regla_distancia = "Empatiza con el desplazamiento y ofrece opciones virtuales."

# Lógica de franjas para las visitas
if visitas < 10:
    regla_visitas = "Anímale a retomar el impulso inicial. Recuérdale que los comienzos cuestan, pero lo importante es volver a dar el primer paso."
elif visitas <= 50:
    regla_visitas = "Menciónale que tenía un ritmo estupendo y anímale a recuperar esa buena rutina de entrenamiento que ya había conseguido."
else:
    regla_visitas = "Hazle saber que valoramos muchísimo su larga y constante trayectoria con nosotros. Anímale a no perder todo ese progreso acumulado."

# 4. Prompt Maestro
prompt = f"""
Actúa como el Director de Fidelización del centro deportivo Can Xaubet.
Escribe un email persuasivo y empático para recuperar al socio con ID {id_socio}.

CONTEXTO DEL SOCIO (INFORMACIÓN INTERNA):
- Días de ausencia: {dias_sin_venir} días.
- Ubicación: {texto_ubicacion}
- Edad: {edad} años.

REGLAS DE REDACCIÓN (IMPORTANTE):
1. SALUDO Y PRIVACIDAD: Usa "Estimado/a socio/a," (prohibido usar el ID). NUNCA menciones la cantidad exacta de días de ausencia.
2. TONO ADAPTADO A LA EDAD: Ajusta la madurez y energía del mensaje sabiendo que tiene {edad} años. 
3. APELA A SU HISTORIAL: {regla_visitas} ESTÁ TERMINANTEMENTE PROHIBIDO mencionar el número exacto de visitas que ha hecho.
4. PERSONALIZACIÓN POR DISTANCIA: {regla_distancia}
5. FLEXIBILIDAD DE CUOTA: Recuérdale que si sus circunstancias han cambiado, estamos a su disposición para adaptar su cuota o tarifa a sus necesidades actuales. NUNCA uses nombres técnicos de cuotas.
6. REGALO: Ofrécele una sesión gratuita de 30 min con un entrenador personal para reevaluar objetivos.
7. CIERRE: Despídete como "Director de Fidelización, Can Xaubet".
8. LÍMITE: Máximo 150 palabras.
"""

print(f"Extrayendo socio al azar de la base de datos...")
print(f"Contexto de la IA: {dias_sin_venir} días sin venir, Zona {zona}, Edad {edad}, Visitas {visitas}, Cuota {cuota}\n")

# 5. Generación del correo (Llamada al modelo local)
try:
    respuesta = client.chat.completions.create(
        model="llama3.1",
        messages=[
            {"role": "system", "content": "Eres un experto en redacción corporativa y retención de clientes."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    print("EMAIL GENERADO ALEATORIAMENTE (VÍA LLAMA 3.1):")
    print("-" * 50)
    print(respuesta.choices[0].message.content)
    print("-" * 50)
    
except Exception as e:
    print(f"Error de conexión: {e}")
    print("Por favor, verifica que LM Studio está abierto y el botón 'Start Server' está activado en la pestaña del servidor local (<->).")

Extrayendo socio al azar de la base de datos...
Contexto de la IA: 70 días sin venir, Zona 0, Edad 75, Visitas 272, Cuota AB. PENSIONISTA

EMAIL GENERADO ALEATORIAMENTE (VÍA LLAMA 3.1):
--------------------------------------------------
Asunto: ¡No pierdas el ritmo! Estamos aquí por ti.

Estimado socio,

Espero que se encuentre bien. Queremos expresarle nuestro agradecimiento por su larga y constante trayectoria en Can Xaubet. Valoramos enormemente su dedicación y compromiso con su salud y bienestar. No queremos verlo perder todo ese progreso acumulado. 

Sabemos que vive muy cerca de nosotros, lo cual es una gran ventaja para poder seguir disfrutando del gimnasio sin problemas. Si sus circunstancias han cambiado desde la última vez que estuvo con nosotros, estamos a su disposición para adaptar su cuota según sus necesidades actuales.

Deseamos ayudarlo a recuperar el ritmo y continuar avanzando en su camino hacia la salud y el bienestar. Por lo tanto, le ofrecemos una sesión gratuita 

In [7]:
import sqlite3
import pandas as pd
from openai import OpenAI
import time

# 1. Conexión al servidor local de Ollama
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# 2. Carga de datos del CRM
conexion = sqlite3.connect('../../datos/crm_gimnasio.db')
df_campana = pd.read_sql("SELECT * FROM campana_retencion", conexion)

lista_emails = []

print(f"--- INICIANDO GENERACIÓN LOCAL MASIVA ---")
print(f"Procesando {len(df_campana)} socios con Llama 3.1...")
print("Esto puede tardar unos minutos dependiendo de la potencia de tu PC...\n")

# 3. Bucle de generación masiva
for index, socio in df_campana.iterrows():
    id_socio = int(socio['id_socio'])
    dias_sin_venir = int(socio['dias_sin_venir'])
    zona = int(socio['zona_proximidad'])
    visitas = int(socio['total_visitas'])
    edad = int(socio['edad'])
    
    # --- LA LÓGICA DETALLADA QUE FUNCIONÓ PERFECTO ---
    
    # Lógica de distancia
    if zona == 0:
        texto_ubicacion = "Vive muy CERCA del gimnasio."
        regla_distancia = "Destaca la comodidad de tener el centro a un paso de casa."
    else:
        texto_ubicacion = "Vive LEJOS del gimnasio."
        regla_distancia = "Empatiza con el desplazamiento y ofrece opciones virtuales."

    # Lógica de franjas para las visitas
    if visitas < 10:
        regla_visitas = "Anímale a retomar el impulso inicial. Recuérdale que los comienzos cuestan, pero lo importante es volver a dar el primer paso."
    elif visitas <= 50:
        regla_visitas = "Menciónale que tenía un ritmo estupendo y anímale a recuperar esa buena rutina de entrenamiento que ya había conseguido."
    else:
        regla_visitas = "Hazle saber que valoramos muchísimo su larga y constante trayectoria con nosotros. Anímale a no perder todo ese progreso acumulado."

    # --- EL PROMPT MAESTRO ---
    prompt = f"""
    Actúa como el Director de Fidelización del centro deportivo Can Xaubet.
    Escribe un email persuasivo y empático para recuperar al socio con ID {id_socio}.

    CONTEXTO DEL SOCIO (INFORMACIÓN INTERNA):
    - Días de ausencia: {dias_sin_venir} días.
    - Ubicación: {texto_ubicacion}
    - Edad: {edad} años.

    REGLAS DE REDACCIÓN (IMPORTANTE):
    1. SALUDO Y PRIVACIDAD: Usa "Estimado/a socio/a," (prohibido usar el ID). NUNCA menciones la cantidad exacta de días de ausencia.
    2. TONO ADAPTADO A LA EDAD: Ajusta la madurez y energía del mensaje sabiendo que tiene {edad} años. 
    3. APELA A SU HISTORIAL: {regla_visitas} ESTÁ TERMINANTEMENTE PROHIBIDO mencionar el número exacto de visitas que ha hecho.
    4. PERSONALIZACIÓN POR DISTANCIA: {regla_distancia}
    5. FLEXIBILIDAD DE CUOTA: Recuérdale que si sus circunstancias han cambiado, estamos a su disposición para adaptar su cuota o tarifa a sus necesidades actuales. NUNCA uses nombres técnicos de cuotas.
    6. REGALO: Ofrécele una sesión gratuita de 30 min con un entrenador personal para reevaluar objetivos.
    7. CIERRE: Despídete como "Director de Fidelización, Can Xaubet".
    8. LÍMITE: Máximo 150 palabras.
    """

    # --- LLAMADA AL MODELO ---
    try:
        respuesta = client.chat.completions.create(
            model="llama3.1", 
            messages=[
                {"role": "system", "content": "Eres un experto en redacción corporativa y retención de clientes."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )
        email_texto = respuesta.choices[0].message.content
        print(f"[OK] -> Correo generado para el Socio {id_socio}")
    except Exception as e:
        email_texto = f"Error en generación: {e}"
        print(f"[ERROR] -> Fallo en el Socio {id_socio}")

    lista_emails.append(email_texto)

# 4. Actualización de la Base de Datos
df_campana['email_propuesto'] = lista_emails
df_campana.to_sql('campana_retencion', conexion, if_exists='replace', index=False)
conexion.close()

print("\n--- ¡PROCESO FINALIZADO CON ÉXITO! ---")
print("Base de datos crm_gimnasio.db actualizada con los 50 correos de IA Local.")

--- INICIANDO GENERACIÓN LOCAL MASIVA ---
Procesando 50 socios con Llama 3.1...
Esto puede tardar unos minutos dependiendo de la potencia de tu PC...

[OK] -> Correo generado para el Socio 4374
[OK] -> Correo generado para el Socio 25538
[OK] -> Correo generado para el Socio 23983
[OK] -> Correo generado para el Socio 27607
[OK] -> Correo generado para el Socio 21984
[OK] -> Correo generado para el Socio 8991
[OK] -> Correo generado para el Socio 23942
[OK] -> Correo generado para el Socio 20800
[OK] -> Correo generado para el Socio 25815
[OK] -> Correo generado para el Socio 28113
[OK] -> Correo generado para el Socio 25537
[OK] -> Correo generado para el Socio 23348
[OK] -> Correo generado para el Socio 14325
[OK] -> Correo generado para el Socio 15794
[OK] -> Correo generado para el Socio 26265
[OK] -> Correo generado para el Socio 20369
[OK] -> Correo generado para el Socio 24218
[OK] -> Correo generado para el Socio 18470
[OK] -> Correo generado para el Socio 12587
[OK] -> Correo 

## Exportación final para Power BI
Una vez generados los correos únicos, exportamos el resultado a CSV para alimentar el Dashboard.

In [8]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect('../../datos/crm_gimnasio.db')
df_final = pd.read_sql("SELECT * FROM campana_retencion", conexion)

ruta_csv = '../../datos/procesados/campana_retencion_final.csv'
df_final.to_csv(ruta_csv, index=False)

conexion.close()
print(f"Archivo exportado correctamente para Power BI en: {ruta_csv}")

Archivo exportado correctamente para Power BI en: ../../datos/procesados/campana_retencion_final.csv
